# Monte Carlo Methods: Comparing Exploration Strategies

This notebook provides a pedagogical demonstration of two different ways to solve the **Exploration vs. Exploitation** problem in Monte Carlo methods:
1. **Monte Carlo ES (Exploring Starts)**: Randomizing the starting point.
2. **On-Policy $\epsilon$-Greedy**: Randomizing the actions from a fixed start.

## 1. Environment and Shared Components

In [ ]:
import numpy as np
import random
from collections import defaultdict

class SimpleGridWorld:
    def __init__(self):
        self.size = 3
        self.goal = 8
    
    def reset(self, start_state=0):
        self.state = start_state
        return self.state
    
    def step(self, action):
        row, col = divmod(self.state, self.size)
        if action == 0: row = max(0, row - 1)    # Up
        elif action == 1: row = min(self.size - 1, row + 1) # Down
        elif action == 2: col = max(0, col - 1)  # Left
        elif action == 3: col = min(self.size - 1, col + 1) # Right
        
        self.state = row * self.size + col
        reward = 10 if self.state == self.goal else -1
        done = (self.state == self.goal)
        return self.state, reward, done

def update_q_values(episode, Q, returns_sum, returns_count, gamma=0.9):
    G = 0
    visited_sa = set()
    for t in range(len(episode) - 1, -1, -1):
        s, a, r = episode[t]
        G = gamma * G + r
        if (s, a) not in visited_sa:
            visited_sa.add((s, a))
            returns_sum[s][a] += G
            returns_count[s][a] += 1
            Q[s][a] = returns_sum[s][a] / returns_count[s][a]

## 2. Method A: Monte Carlo ES (Exploring Starts)

**Logic**: We use a purely **Greedy** policy, but we force exploration by starting the agent in a **RANDOM** state every time.

In [ ]:
def run_mc_es(num_iterations=20):
    env = SimpleGridWorld()
    Q = defaultdict(lambda: np.zeros(4))
    returns_sum = defaultdict(lambda: np.zeros(4))
    returns_count = defaultdict(lambda: np.zeros(4))
    policy = defaultdict(lambda: 3) # Initial guess: All Right

    print("--- Running Monte Carlo ES (Random Starts) ---")
    for i in range(num_iterations):
        # EXPLORING START: Pick random state and random FIRST action
        s0 = random.choice(range(8))
        a0 = random.choice(range(4))
        
        episode = []
        state = env.reset(s0)
        
        # First step is random
        next_state, reward, done = env.step(a0)
        episode.append((state, a0, reward))
        state = next_state
        
        # Rest of episode follows Greedy Policy
        while not done:
            action = policy[state]
            next_state, reward, done = env.step(action)
            episode.append((state, action, reward))
            state = next_state
            
        update_q_values(episode, Q, returns_sum, returns_count)
        
        # Improvement: Greedy
        for s in range(8):
            policy[s] = np.argmax(Q[s])
            
    print("Final Policy Grid (ES):")
    print(np.array([policy[s] for s in range(9)]).reshape(3,3))

run_mc_es()

## 3. Method B: On-Policy $\epsilon$-Greedy MC

**Logic**: We start from a **FIXED** state (State 0), but we force exploration by taking **RANDOM ACTIONS** with probability $\epsilon$.

In [ ]:
def run_mc_epsilon_greedy(num_iterations=100, epsilon=0.2):
    env = SimpleGridWorld()
    Q = defaultdict(lambda: np.zeros(4))
    returns_sum = defaultdict(lambda: np.zeros(4))
    returns_count = defaultdict(lambda: np.zeros(4))
    
    # Note: In epsilon-greedy, the 'policy' is stochastic, 
    # so we often just derive it from Q during action selection.

    print(f"--- Running Epsilon-Greedy MC (Fixed Start, eps={epsilon}) ---")
    for i in range(num_iterations):
        # FIXED START: Always start at State 0
        state = env.reset(0) 
        episode = []
        done = False
        
        while not done:
            # EPSILON-GREEDY ACTION SELECTION
            if random.random() < epsilon:
                action = random.choice(range(4)) # Explore
            else:
                action = np.argmax(Q[state]) # Exploit
                
            next_state, reward, done = env.step(action)
            episode.append((state, action, reward))
            state = next_state
            
            # Safety break for long episodes in early training
            if len(episode) > 100: break
            
        update_q_values(episode, Q, returns_sum, returns_count)
        
    print("Final Policy Grid (Epsilon-Greedy Derive from Q):")
    final_policy = [np.argmax(Q[s]) for s in range(9)]
    print(np.array(final_policy).reshape(3,3))

run_mc_epsilon_greedy()

## 4. Key Takeaways

1. **Monte Carlo ES**: Learns very fast because it can 'teleport' to the goal. However, it is impossible to use in real-world systems like self-driving cars.
2. **$\epsilon$-Greedy**: More realistic. The agent starts from the beginning and must 'stumble' upon the goal by accident first. Once it finds the goal, it slowly refines its path by reducing randomness.